In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors

In [ ]:
qm8 = pd.read_csv("C:\\Users\\Diya\\PBL\\Molecular-Property-Prediction-Using-GNN-and-Transformer\\Dataset\\qm8.csv")
qm9 = pd.read_csv("C:\\Users\\Diya\\PBL\\Molecular-Property-Prediction-Using-GNN-and-Transformer\\Dataset\\qm9.csv")

In [ ]:
qm8.head(15)

In [ ]:
qm9.head(15)

In [ ]:
qm8.shape

In [ ]:
qm9.shape

In [ ]:
qm8.describe()

In [ ]:
qm9.describe()

In [ ]:
qm8.info()

In [ ]:
qm9.info()

In [ ]:
qm9.isnull().sum()

In [ ]:
qm8.isnull().sum()

In [ ]:
print("QM8",qm8["smiles"].duplicated().sum())
print("Qm8",qm9["smiles"].duplicated().sum())


In [ ]:
print(qm8.isna().mean()*100)
print(qm9.isna().mean()*100)


In [ ]:
qm8.columns

In [ ]:
qm9.columns

In [ ]:
qm8 = qm8.drop_duplicates(subset="smiles", keep="first")
qm9 = qm9.drop_duplicates(subset="smiles", keep="first")



In [ ]:
qm8.shape

In [ ]:
qm9.shape

In [ ]:
print(qm8["smiles"].duplicated().sum())
print(qm9["smiles"].duplicated().sum())


In [ ]:
#Checking Overlaping

qm8_smiles = set(qm8["smiles"])
qm9_smiles = set(qm9["smiles"])

print("Common molecules:", len(qm8_smiles & qm9_smiles))
print("QM8 only:", len(qm8_smiles - qm9_smiles))
print("QM9 only:", len(qm9_smiles - qm8_smiles))


In [ ]:
merge_df = pd.merge(qm8,qm9,on="smiles",how="outer")


In [ ]:
merge_df.head(15)

In [ ]:
merge_df.shape

In [ ]:
def valid_smiles(s):
    return Chem.MolFromSmiles(s) is not None

merge_df = merge_df[merge_df["smiles"].apply(valid_smiles)]


In [ ]:
merge_df = merge_df.drop_duplicates(subset="smiles")


In [ ]:
merge_df.shape

In [ ]:
print("Total molecules:", merge_df.shape[0])
print("Total properties:", merge_df.shape[1] - 1)


In [ ]:
target_cols = merge_df.select_dtypes(include=["float64"]).columns
merge_df[target_cols].isna().mean().sort_values(ascending=False).head(20)

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(merge_df.isna(),cmap="Blues", cbar=False)
plt.title("Missing Value Map")
plt.show()

In [ ]:
missing_pct = (merge_df.isna().mean()*100).sort_values(ascending=False)
missing_pct


In [ ]:
plt.figure(figsize=(12,8))
numeric_df = merge_df.select_dtypes(include=["number"])
sns.heatmap(numeric_df.corr(),cmap="coolwarm",annot=False)
plt.title("Correlation Heatmap of Molecular Properties")
plt.show()

In [ ]:
merge_df["num_atoms"] = merge_df["smiles"].apply(lambda x: Chem.MolFromSmiles(x).GetNumAtoms())
merge_df["num_atoms"]

In [ ]:
merge_df["num_bonds"] = merge_df["smiles"].apply(lambda x: Chem.MolFromSmiles(x).GetNumBonds())
merge_df["num_bonds"]

In [ ]:
merge_df["smiles_len"] = merge_df["smiles"].apply(len)
merge_df["smiles_len"]

In [ ]:
#Different strings can represent same molecule so converting

def canonical(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol)

merge_df["smiles"] = merge_df["smiles"].apply(canonical)


In [ ]:
merge_df = merge_df.drop_duplicates(subset="smiles")


In [ ]:
merge_df.columns = merge_df.columns.str.strip() 

In [ ]:
y = merge_df.drop(columns=["smiles"])

# Convert all columns to numeric
y = y.apply(pd.to_numeric, errors="coerce")

# Temporary fill NaNs
y_temp = np.nan_to_num(y.values, nan=0.0)

# Convert to tensor
import torch
labels_tensor = torch.tensor(y_temp,dtype=torch.float32)

# Standardize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
y_scaled = scaler.fit_transform(y_temp)

# Back to DataFrame

y_scaled = pd.DataFrame(y_scaled,columns=y.columns)

In [ ]:
y_scaled

In [ ]:
x = merge_df["smiles"]
y = y_scaled

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y_scaled,test_size=0.2,random_state=42)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv,GlobalAttention

In [ ]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),            #element type for chemical identity
        atom.GetDegree(),               #bonds count for reactivity
        atom.GetFormalCharge(),        #charge for polarity
        atom.GetHybridization().real,  #orbital type - geometry
        atom.GetTotalNumHs(),           #attached H - saturation
        atom.GetIsAromatic(),          #ring structural for stability
        atom.GetMass(),                #atomic weight 
        atom.GetNumRadicalElectrons()  #unpaired electrons - reactivity
    ]

def bond_features(bond):
    return [
        bond.GetBondTypeAsDouble(),    #single/double/triple
        bond.GetIsConjugated(),       #electron sharing
        bond.IsInRing()              #cyclic structure
    ]

In [ ]:
#smiles to graph conversion

def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)      #smiles to molecule object
    node_feats = [atom_features(atom)
        for atom in mol.GetAtoms()        #builds feature matrix
    ]

    edge_index = []                    #stores connection

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]       

    x = torch.tensor(node_feats, dtype=torch.float)      #node features to tensor

    edge_index = torch.tensor(edge_index,dtype=torch.long).t().contiguous()    #edges to tensor

    y = torch.tensor(label,dtype=torch.float).unsqueeze(0)    #unsqueeze coverts (39,) to (1,39)

    return Data(x=x,edge_index=edge_index,y=y)

In [ ]:
#converting each molecule into graph

train_graphs = [smiles_to_graph(x_train.iloc[i],y_train.iloc[i].values)
    for i in range(len(x_train))]


test_graphs = [smiles_to_graph(x_test.iloc[i],y_test.iloc[i].values)
    for i in range(len(x_test))]

In [ ]:
#Batches graphs , Enables GPU training , Shuffles for generalization

train_loader = DataLoader(train_graphs,batch_size=32,shuffle=True)
test_loader = DataLoader(test_graphs,batch_size=32)

In [ ]:
#Defines neural network inside each GIN layer

def gin_mlp(in_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, out_dim),
        nn.ReLU(),
        nn.Linear(out_dim, out_dim)
    )

In [ ]:
num_features = train_graphs[0].x.shape[1]  #Auto-detects atom feature size
out_dim = y_train.shape[1]                 #Number of properties predicted

In [ ]:
def masked_mse(pred, target):

    mask = ~torch.isnan(target)       #Identifies valid labels
    loss = (pred - target) ** 2
    loss = loss * mask                 #Ignores missing labels
    return loss.sum() / mask.sum()